# เทียบฐานข้อมูลของเรา กับ The Perovskite Database (NOMAD)

**คำถาม 2 ข้อ**
1. หลังกรอง NOMAD ด้วยเงื่อนไขด้านล่างแล้ว มีเปเปอร์ที่ **ซ้ำ** กับฐานข้อมูลของเรากี่เปเปอร์
2. ฐานข้อมูลของเรามี **จุดแข็ง** อะไรเมื่อเทียบกับ NOMAD — ตอบด้วยตัวเลขที่นับได้ ไม่ใช่ความรู้สึก

**เงื่อนไขกรอง** (ตรงกับเมนูในหน้า GUI ของ NOMAD — ใช้ช่องข้อมูลเดียวกันเป๊ะ)

| # | เงื่อนไข | เมนูใน GUI | ช่องข้อมูลที่ใช้กรอง |
|---|---|---|---|
| 1 | A-site = Cs; FA; MA | Perovskite Material → A cations | `data.perovskite.composition_a_ions` |
| 2 | HTL = NiOx หรือ NiO-c หรือ NiO-np | Transport Layers → Hole transport layer | `results...solar_cell.hole_transport_layer` |
| 3 | โครงสร้าง pin | Device Architecture → architecture | `data.cell.architecture` |
| 4 | มี Efficiency **หรือ** T80 (ถ้าไม่มี T80 ใช้ PCE initial + PCE end แทน) | Solar Cell Performance / Stability | กรองในเครื่องเราหลังดาวน์โหลด |

เงื่อนไข 1–3 ให้เซิร์ฟเวอร์ NOMAD กรองให้ · เงื่อนไข 4 เป็นแบบ "มีค่าหรือไม่" ซึ่งกรองเองหลังดาวน์โหลดง่ายกว่า

**ข้อมูลของเรา** = `processed/prepared_v2/` (ผลจาก `run_pipeline.py`) · **ข้อมูล NOMAD** ดึงสดผ่าน API แล้วเก็บ snapshot ไว้ใน `notebooks/nomad_lab/cache/`
(NOMAD เป็นฐานข้อมูลที่ยังโตอยู่ — ตัวเลขในโน้ตบุ๊กนี้คือ ณ วันที่ของ snapshot ที่พิมพ์ไว้ในเซลล์ดาวน์โหลด · ลบโฟลเดอร์ cache ถ้าอยากดึงใหม่)

## คำศัพท์

| คำ | ความหมายแบบภาษาคน |
|---|---|
| **NOMAD / The Perovskite Database** | ฐานข้อมูลสาธารณะของยุโรปที่รวมเซลล์ perovskite จากเปเปอร์ทั่วโลก (~5 หมื่นเครื่อง) — ชุดดั้งเดิมคนกรอกเอง (ถึงปี 2020) ช่วงหลังเพิ่มด้วย **LLM** อ่านเปเปอร์แล้วกรอกให้อัตโนมัติ |
| **เครื่อง (device)** | เซลล์แสงอาทิตย์ 1 ชิ้นที่เปเปอร์รายงานผล — 1 เปเปอร์มักมีหลายเครื่อง (เครื่องควบคุม + เครื่องที่ปรับปรุง) |
| **DOI** | รหัสประจำเปเปอร์ เช่น `10.1002/smtd.202200787` — ใช้จับคู่ว่าเปเปอร์ซ้ำกันไหม |
| **A-site: Cs / FA / MA** | ไอออนบวก 3 ชนิดในผลึก perovskite (ซีเซียม / ฟอร์มามิดิเนียม / เมทิลแอมโมเนียม) — "Cs; FA; MA" = สูตรที่ผสมครบทั้งสาม (triple cation) |
| **HTL, NiOx / NiO-c / NiO-np** | ชั้นส่งโฮล (ประจุบวก) ทำจากนิกเกิลออกไซด์ — NOMAD แยกชื่อตามรูปแบบฟิล์ม: c = ฟิล์มทึบ (compact), np = อนุภาคนาโน, x = ไม่ระบุ |
| **pin** | ลำดับชั้นแบบ "กลับหัว" (inverted): HTL อยู่ล่าง perovskite ETL อยู่บน — ฐานข้อมูลเราทั้งหมดเป็นแบบนี้ |
| **PCE (Efficiency)** | ประสิทธิภาพแปลงแสงเป็นไฟฟ้า (%) |
| **Voc, Jsc, FF** | แรงดันวงจรเปิด (V), กระแสลัดวงจร (mA/cm²), fill factor — PCE ≈ Voc × Jsc × FF |
| **T80** | เวลา (ชั่วโมง) ที่เซลล์เสื่อมจนเหลือ 80% ของประสิทธิภาพตั้งต้น — ยิ่งนานยิ่งทน |
| **retention** | ประสิทธิภาพที่เหลือ (% ของตั้งต้น) หลังทดสอบไปนานเท่าที่กำหนด |
| **เครื่องควบคุม (control)** | เครื่องที่ไม่ได้ปรับปรุงอะไร ในเปเปอร์เดียวกัน — ใช้เป็นฐานคิด Δ (ส่วนต่างที่การปรับปรุงทำได้) |

In [1]:
import json
import math
import re
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests
from perovskite_solar_cell_database.schema import PerovskiteSolarCell
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "processed").exists())
P = ROOT / "processed/prepared_v2"
CACHE = ROOT / "notebooks/nomad_lab/cache"
CACHE.mkdir(exist_ok=True)

API = "https://nomad-lab.eu/prod/v1/api/v1"
SCHEMA = PerovskiteSolarCell.m_def.qualified_name()
SC = "results.properties.optoelectronic.solar_cell"

session = requests.Session()
session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=8, backoff_factor=3, status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["POST"], respect_retry_after_header=True)))

pd.set_option("display.max_colwidth", 70)
pd.set_option("display.width", 220)
print("schema:", SCHEMA)

C:\Users\acer\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


schema: perovskite_solar_cell_database.schema.PerovskiteSolarCell


## 1. ดึงข้อมูล NOMAD (เงื่อนไข 1–3 กรองที่เซิร์ฟเวอร์)

- ชื่อ schema (`PerovskiteSolarCell`) เอามาจากไลบรารี `perovskite-solar-cell-database` โดยตรง ไม่ได้พิมพ์เอง
- ยิงไปที่ endpoint `entries/archive/query` ตัวเดียวกับที่ `ArchiveQuery` ของ `nomad-lab` ใช้ แต่เรียกผ่าน `requests` เอง
  เพราะ `ArchiveQuery` **ล่มทั้งก้อนเมื่อเซิร์ฟเวอร์ตัดการเชื่อมต่อ** (เจอจริงตอนทดลอง) และของที่โหลดมาแล้วหายหมด — ที่นี่ retry ให้อัตโนมัติ
- ขอเฉพาะส่วน `data` ที่ใช้ (ตัด metadata ดัชนีค้นหาและกราฟ plotly ทิ้ง) — จาก 62 MB เหลือราว 4 MB
- ครั้งแรกใช้เวลา ~3 นาที (API จำกัดความถี่) ครั้งต่อไปอ่านจาก cache ทันที

In [2]:
FILTER_1_3 = {
    "section_defs.definition_qualified_name:all": [SCHEMA],
    f"data.perovskite.composition_a_ions#{SCHEMA}": "Cs; FA; MA",
    f"{SC}.hole_transport_layer:any": ["NiOx", "NiO-c", "NiO-np"],
    f"data.cell.architecture#{SCHEMA}": "pin",
}
SECTIONS = ["ref", "cell", "substrate", "htl", "perovskite", "perovskite_deposition", "etl",
            "backcontact", "add", "encapsulation", "jv", "stabilised", "eqe", "stability"]
REQUIRED = {
    "metadata": {"entry_id": "*", "upload_id": "*"},
    "data": {s: "*" for s in SECTIONS},
    "results": {"properties": {"optoelectronic": "*"}},
}


def download_archives(query, page_size=25):
    body = {"owner": "visible", "query": query, "required": REQUIRED,
            "pagination": {"page_size": page_size}}
    out = []
    while True:
        r = session.post(f"{API}/entries/archive/query", json=body, timeout=300)
        r.raise_for_status()
        j = r.json()
        out += [d["archive"] for d in j["data"]]
        after = j["pagination"].get("next_page_after_value")
        if not after or not j["data"]:
            return out
        body["pagination"]["page_after_value"] = after


def all_nomad_dois():
    rows, after = [], None
    while True:
        pag = {"page_size": 10000, **({"page_after_value": after} if after else {})}
        body = {"owner": "visible",
                "query": {"section_defs.definition_qualified_name:all": [SCHEMA]},
                "pagination": {"page_size": 0},
                "aggregations": {"doi": {"terms": {"quantity": f"data.ref.DOI_number#{SCHEMA}",
                                                   "pagination": pag}}}}
        r = session.post(f"{API}/entries/query", json=body, timeout=300)
        r.raise_for_status()
        t = r.json()["aggregations"]["doi"]["terms"]
        rows += [[b["value"], b["count"]] for b in t["data"]]
        after = t.get("pagination", {}).get("next_page_after_value")
        if not after or not t["data"]:
            return rows


def cached(name, fetch):
    f = CACHE / name
    if f.exists():
        return json.loads(f.read_text())
    snap = {"downloaded_at": datetime.now(timezone.utc).isoformat(timespec="seconds"), "items": fetch()}
    f.write_text(json.dumps(snap))
    return snap


snap = cached("nomad_csfama_nio_pin.json", lambda: download_archives(FILTER_1_3))
archives = snap["items"]
snap_dois = cached("nomad_all_dois.json", all_nomad_dois)
nomad_dois = pd.DataFrame(snap_dois["items"], columns=["doi", "n_devices"])

print(f"snapshot: {snap['downloaded_at']}")
print(f"NOMAD ทั้งฐาน: {nomad_dois.n_devices.sum():,} เครื่องที่มี DOI · {len(nomad_dois):,} ชื่อ DOI")
print(f"ผ่านเงื่อนไข 1–3: {len(archives)} เครื่อง")

snapshot: 2026-10-12T05:49:52+00:00
NOMAD ทั้งฐาน: 51,356 เครื่องที่มี DOI · 10,250 ชื่อ DOI
ผ่านเงื่อนไข 1–3: 394 เครื่อง


## 2. แปลงเป็นตาราง 1 แถว = 1 เครื่อง

ข้อควรรู้ของข้อมูล NOMAD ที่ต้องจัดการก่อน:
- ค่าที่ไม่รู้ถูกเขียนเป็นข้อความ `"Unknown"`, `"nan; nan"` หรือปนกันเช่น `"Unknown | 100"` (ชั้นแรกไม่รู้ ชั้นที่สองรู้) —
  ฟังก์ชัน `known()` นับว่า "มีข้อมูล" ถ้ามีอย่างน้อยหนึ่งส่วนที่ไม่ใช่ค่าว่าง
- `Jsc` ในส่วน `results` เก็บเป็นหน่วย SI **A/m²** → หาร 10 ให้เป็น mA/cm² แบบที่เปเปอร์ใช้ · `FF` เป็นสัดส่วน 0–1 (ของเราเป็น %)
- DOI เขียนหลายแบบ (`https://www.doi.org/10.…`, มีอักขระ BOM นำหน้า) → แปลงเป็น `doi_key` ด้วยกติกาเดียวกับ
  `pipeline/stage1_normalize.py` (ตัวพิมพ์เล็ก ตัดทุกอย่างที่ไม่ใช่ a-z0-9) หลังตัด prefix ออก

In [3]:
MISSING = {"", "unknown", "nan", "none", "n/a", "na"}


def known(v):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return False
    if isinstance(v, (bool, int, float)):
        return True
    return any(p.strip().lower() not in MISSING for p in re.split(r"\||;|>>", str(v)))


def get(arc, path):
    cur = arc
    for k in path.split("."):
        if not isinstance(cur, dict):
            return None
        cur = cur.get(k)
    return cur


def num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return math.nan


def doi_key(s):
    s = str(s).replace("﻿", "").strip().lower()
    m = re.search(r"10[._]\d{4,9}.*", s)
    return re.sub(r"[^0-9a-z]", "", m.group(0) if m else s)


SUPPLIER = ["substrate.supplier", "htl.deposition_reaction_solutions_compounds_supplier",
            "htl.deposition_solvents_supplier", "perovskite_deposition.reaction_solutions_compounds_supplier",
            "perovskite_deposition.solvents_supplier", "etl.deposition_reaction_solutions_compounds_supplier",
            "etl.deposition_solvents_supplier", "backcontact.deposition_reaction_solutions_compounds_supplier"]
PURITY = [p.replace("supplier", "purity") for p in SUPPLIER if p != "substrate.supplier"]


def flatten(a):
    d = a["data"]
    sc = get(a, SC) or {}
    return {
        "entry_id": a["metadata"]["entry_id"],
        "doi": get(d, "ref.DOI_number"),
        "year": pd.to_datetime(get(d, "ref.publication_date"), errors="coerce", utc=True).year,
        "method": get(d, "ref.extraction_method") or "manual",
        "sample_id": get(d, "ref.internal_sample_id"),
        "a_ions": get(d, "perovskite.composition_a_ions"),
        "composition": get(d, "perovskite.composition_long_form"),
        "htl_list": sc.get("hole_transport_layer"),
        "htl_stack": get(d, "htl.stack_sequence"),
        "htl_additive": get(d, "htl.additives_compounds"),
        "architecture": get(d, "cell.architecture"),
        "illum": num(sc.get("illumination_intensity")),
        "pce": num(sc.get("efficiency")),
        "voc": num(sc.get("open_circuit_voltage")),
        "jsc": num(sc.get("short_circuit_current_density")) / 10,
        "ff": num(sc.get("fill_factor")),
        "rev_pce": num(get(d, "jv.reverse_scan_PCE")),
        "fwd_pce": num(get(d, "jv.forward_scan_PCE")),
        "eqe_jsc": num(get(d, "eqe.integrated_Jsc")),
        "pl_max": get(d, "perovskite.pl_max"),
        "sub_rms": get(d, "substrate.surface_roughness_rms"),
        "t80_h": num(get(d, "stability.PCE_T80")),
        "pce_init": num(get(d, "stability.PCE_initial_value")),
        "pce_end": num(get(d, "stability.PCE_end_of_experiment")),
        "stab_time_h": num(get(d, "stability.time_total_exposure")),
        "stab_protocol": get(d, "stability.protocol"),
        "stab_temp": get(d, "stability.temperature_range"),
        "stab_rh": get(d, "stability.relative_humidity_range"),
        "stab_light": get(d, "stability.light_intensity"),
        "stab_atm": get(d, "stability.atmosphere"),
        "htl_anneal_T": get(d, "htl.deposition_thermal_annealing_temperature"),
        "htl_anneal_t": get(d, "htl.deposition_thermal_annealing_time"),
        "psk_anneal_T": get(d, "perovskite_deposition.thermal_annealing_temperature"),
        "psk_anneal_t": get(d, "perovskite_deposition.thermal_annealing_time"),
        "psk_conc": get(d, "perovskite_deposition.reaction_solutions_concentrations"),
        "quench_vol": get(d, "perovskite_deposition.quenching_media_volume"),
        "psk_thick": get(d, "perovskite.thickness"),
        "htl_thick": get(d, "htl.thickness_list"),
        "supplier": next((get(d, p) for p in SUPPLIER if known(get(d, p))), None),
        "purity": next((get(d, p) for p in PURITY if known(get(d, p))), None),
    }


nm = pd.DataFrame([flatten(a) for a in archives])
nm["doi_key"] = nm["doi"].map(doi_key)
nm[["doi", "year", "method", "composition", "htl_stack", "pce", "voc", "jsc", "ff", "t80_h", "pce_init", "pce_end"]].head(8)

,doi,year,method,composition,htl_stack,pce,voc,jsc,ff,t80_h,pce_init,pce_end
0,https://www.doi.org/10.1002/adma.202413304,2024,LLM,Cs0.05FA0.9025MA0.0475PbBr0.15I2.85,NiOx | Me-4PAcz,21.82,1.110,24.760,0.7886,NaN,NaN,NaN
1,10.1021/acsaem.9b01200,2019,manual,Cs0.05FA0.81MA0.14PbBr0.45I2.55,NiO-c,17.05,1.130,20.530,0.7400,NaN,NaN,72.0
2,https://www.doi.org/10.1002/aenm.202500572,2025,LLM,Cs0.05FA0.9025MA0.0475PbBr0.06I2.94,NiOx | 2PACz-3-UA,25.00,NaN,NaN,NaN,NaN,NaN,NaN
3,10.1021/acsenergylett.9b00135,2019,manual,Cs0.05FA0.85MA0.1PbBr0.5I2.5,NiO-c | PTAA,13.60,1.030,19.600,0.6720,NaN,NaN,NaN
4,10.1039/c9ta08791e,2019,manual,Cs0.05FA0.8075MA0.1425PbBr0.45I2.55,NiO-c | PTAA,18.15,1.116,20.678,0.7860,NaN,NaN,NaN
5,10.1039/c7ta07775k,2017,manual,Cs0.05FA0.79MA0.16PbBr0.51I2.49,NiO-c | NiO-mp,2.67,0.920,19.150,0.1520,NaN,NaN,NaN
6,10.1002/aenm.201903184,2019,manual,Cs0.1FA0.75MA0.15PbBr0.45I2.55,NiO-c,15.00,1.070,20.400,0.6800,NaN,NaN,NaN
7,https://www.doi.org/10.1016/j.mtcomm.2025.113340,2025,LLM,Cs0.05FA0.83MA0.17PbI3,NiOx,21.74,1.120,25.010,0.7761,NaN,NaN,NaN


**อ่านยังไง:** แต่ละแถวคือเซลล์ 1 ชิ้นใน NOMAD ที่ผ่านเงื่อนไข 1–3 แล้ว · คอลัมน์ `method` บอกว่าใครกรอก —
`manual` = คนกรอก (ชุดดั้งเดิม), `LLM` = โมเดลภาษาอ่านเปเปอร์แล้วกรอกให้ ซึ่งสำคัญมากในหัวข้อจุดแข็ง (ข้อ 4.2)

## 3. เงื่อนไข 4 — ต้องมีค่าประสิทธิภาพหรือค่าความเสถียร

ตีความตามที่สั่งตรงตัว: ผ่านถ้า **มี Efficiency** หรือ **มี T80** หรือ **มีทั้ง PCE initial และ PCE end** (กรณีไม่มี T80)
ตารางด้านล่างแยกให้เห็นว่าแต่ละเงื่อนไขย่อยมีกี่เครื่อง — ถ้าอยากได้แบบเข้มกว่า (ต้องมีข้อมูลความเสถียรเท่านั้น) ดูแถวสุดท้าย

In [4]:
c_eff = nm["pce"].notna()
c_t80 = nm["t80_h"].notna()
c_init_end = nm["pce_init"].notna() & nm["pce_end"].notna()
nm["pass4"] = c_eff | c_t80 | c_init_end
nm["has_stab"] = c_t80 | c_init_end

cond4 = pd.DataFrame({
    "เงื่อนไขย่อย": ["มี Efficiency", "มี T80", "ไม่มี T80 แต่มี PCE initial + PCE end", "ผ่านเงื่อนไข 4 (Efficiency หรือ T80 หรือ initial+end)",
                     "(แบบเข้ม) มีข้อมูลความเสถียร: T80 หรือ initial+end"],
    "เครื่อง": [c_eff.sum(), c_t80.sum(), (~c_t80 & c_init_end).sum(), nm["pass4"].sum(), nm["has_stab"].sum()],
    "เปเปอร์": [nm.loc[m, "doi_key"].nunique() for m in [c_eff, c_t80, ~c_t80 & c_init_end, nm["pass4"], nm["has_stab"]]],
})
nmf = nm[nm["pass4"]].copy()
cond4

,เงื่อนไขย่อย,เครื่อง,เปเปอร์
0,มี Efficiency,388,121
1,มี T80,12,6
2,ไม่มี T80 แต่มี PCE initial + PCE end,44,32
3,ผ่านเงื่อนไข 4 (Efficiency หรือ T80 หรือ initial+end),389,121
4,(แบบเข้ม) มีข้อมูลความเสถียร: T80 หรือ initial+end,56,37


**อ่านยังไง:** แทบทุกเครื่องใน NOMAD มีค่า Efficiency อยู่แล้ว เงื่อนไข 4 แบบตรงตัวจึงตัดออกน้อยมาก —
ชุดที่ใช้ต่อจากนี้ (`nmf`) คือแถว "ผ่านเงื่อนไข 4" · แต่ถ้าดูเฉพาะเครื่องที่ **มีข้อมูลความเสถียร** จะเหลือนิดเดียว (แถวสุดท้าย)
ซึ่งเป็นประเด็นหลักของข้อ 4.2

## 4. งานที่ 1 — เปเปอร์ที่ซ้ำกับฐานข้อมูลของเรา

### 4.0 ฐานข้อมูลของเรา
ชื่อไฟล์ Excel บางเปเปอร์เขียน DOI สองแบบ (เช่น `10.1002_solr.202400534` กับ `10_1002_solr_202400534`) — นับด้วย `doi_key` จะได้จำนวนเปเปอร์จริง

In [5]:
dev = pd.read_csv(P / "devices.csv")
tt = pd.read_csv(P / "target_table.csv")
ours = dev[["device_id", "doi_raw"]].merge(tt, on="device_id")
ours["doi_key"] = ours["doi_raw"].map(doi_key)

papers_ours = ours.groupby("doi_key").agg(
    doi_ours=("doi_raw", lambda s: " / ".join(sorted(set(s)))),
    year=("year", "first"),
    n_ours=("device_id", "size"),
    pce_max_ours=("pce", "max"),
)
spelled_twice = papers_ours[papers_ours["doi_ours"].str.contains(" / ")]
comp = pd.read_csv(P / "psk_composition.csv").set_index("device_id")
triple = (comp[["a_Cs", "a_FA", "a_MA"]].fillna(0) > 0).all(axis=1)
print(f"ฐานข้อมูลเรา: {len(ours)} เครื่อง · {ours.doi_raw.nunique()} ชื่อ DOI ในไฟล์ → {len(papers_ours)} เปเปอร์จริง")
print(f"เครื่องที่เป็น Cs/FA/MA ครบสาม: {int(triple.sum())}/{len(comp)} · HTL เป็น NiOx และโครงสร้าง pin ทุกเครื่อง")
print(f"เปเปอร์ที่ชื่อไฟล์เขียน DOI สองแบบ: {len(spelled_twice)}")
spelled_twice[["doi_ours", "n_ours"]]

ฐานข้อมูลเรา: 134 เครื่อง · 45 ชื่อ DOI ในไฟล์ → 43 เปเปอร์จริง
เครื่องที่เป็น Cs/FA/MA ครบสาม: 124/134 · HTL เป็น NiOx และโครงสร้าง pin ทุกเครื่อง
เปเปอร์ที่ชื่อไฟล์เขียน DOI สองแบบ: 2


,doi_ours,n_ours
doi_key,,
101002solr202400534,10.1002_solr.202400534 / 10_1002_solr_202400534,4
101021acsami5c16972,10.1021_acsami.5c16972 / 10_1021_acsami_5c16972,6


**อ่านยังไง:** สองเปเปอร์นี้ถูกกรอก **ซ้ำสองชุด** (คนละชื่อไฟล์ เครื่องเดียวกัน) — `10.1002/solr.202400534` 2+2 เครื่อง และ
`10.1021/acsami.5c16972` 3+3 เครื่อง รวม 5 เครื่องที่นับเกิน · และชุดของ `solr.202400534` สองชุด **บันทึกสัดส่วน Br ไม่ตรงกัน**
(0.03 กับ 0.17) ต้องเปิดเปเปอร์ตัดสินว่าชุดไหนถูกแล้วลบอีกชุด (pipeline แจ้งไว้แล้วใน `processed/review/review_queue.csv` หัวข้อ doi-spelling-dup)
โน้ตบุ๊กนี้นับระดับ **เปเปอร์** ด้วย `doi_key` จึงไม่กระทบคำตอบข้อ 1 · ส่วนตัวเลขระดับเครื่องในข้อ 5 เกินจริงอยู่ 5 เครื่อง (ไม่เปลี่ยนข้อสรุป)

### 4.1 ซ้ำกับชุดที่กรองแล้ว (ตอบคำถามข้อ 1 ตรง ๆ)

รายงานแยก **manual** กับ **LLM** เพราะสองชุดนี้ครอบคลุมคนละช่วงปีและผลิตข้อมูลคนละวิธี
ตัวเลขรวมอย่างเดียวจะกลบข้อเท็จจริงสำคัญไว้ (ดูบรรทัดสรุปใต้ผลลัพธ์)

In [6]:
papers_nm = nmf.groupby("doi_key").agg(
    doi_nomad=("doi", "first"),
    n_nomad=("entry_id", "size"),
    pce_max_nomad=("pce", "max"),
    method=("method", lambda s: ", ".join(sorted(set(s)))),
)
overlap = papers_ours.join(papers_nm, how="inner")
overlap["pce_diff"] = (overlap["pce_max_nomad"] - overlap["pce_max_ours"]).round(2)

man_papers = set(nmf.loc[nmf["method"] == "manual", "doi_key"])
llm_papers = set(nmf.loc[nmf["method"] == "LLM", "doi_key"])
ov_man = set(papers_ours.index) & man_papers
ov_llm = set(papers_ours.index) & llm_papers

print(f"NOMAD หลังกรองครบ 4 เงื่อนไข: {len(nmf)} เครื่อง · {len(papers_nm)} เปเปอร์")
print(f"ซ้ำกับฐานข้อมูลเรา: {len(overlap)} เปเปอร์ จาก {len(papers_ours)} ของเรา "
      f"({len(overlap) / len(papers_ours):.0%}) · คิดเป็น {len(overlap) / len(papers_nm):.0%} ของ NOMAD ชุดนี้")
print(f"  ชุด manual: ซ้ำ {len(ov_man)} เปเปอร์ (จาก {len(man_papers)} เปเปอร์ในชุด)")
print(f"  ชุด LLM   : ซ้ำ {len(ov_llm)} เปเปอร์ (จาก {len(llm_papers)} เปเปอร์ในชุด)")
print(f"  -> การซ้ำแทบทั้งหมดอยู่ฝั่ง LLM เพราะชุด manual หยุดที่ปี "
      f"{int(nmf.loc[nmf['method'] == 'manual', 'year'].max())} "
      f"ส่วนเปเปอร์ของเราเริ่มที่ปี {int(papers_ours['year'].min())}")
overlap[["doi_nomad", "year", "n_ours", "n_nomad", "method", "pce_max_ours", "pce_max_nomad", "pce_diff"]].sort_values("year")

NOMAD หลังกรองครบ 4 เงื่อนไข: 389 เครื่อง · 121 เปเปอร์
ซ้ำกับฐานข้อมูลเรา: 7 เปเปอร์ จาก 43 ของเรา (16%) · คิดเป็น 6% ของ NOMAD ชุดนี้
  ชุด manual: ซ้ำ 1 เปเปอร์ (จาก 47 เปเปอร์ในชุด)
  ชุด LLM   : ซ้ำ 6 เปเปอร์ (จาก 74 เปเปอร์ในชุด)
  -> การซ้ำแทบทั้งหมดอยู่ฝั่ง LLM เพราะชุด manual หยุดที่ปี 2021 ส่วนเปเปอร์ของเราเริ่มที่ปี 2017


,doi_nomad,year,n_ours,n_nomad,method,pce_max_ours,pce_max_nomad,pce_diff
doi_key,,,,,,,,
101007s1085401780949,10.1007/s10854-017-8094-9,2017.0,4,10,manual,12.80,12.80,0.00
101002smtd202200787,https://www.doi.org/10.1002/smtd.202200787,2022.0,2,2,LLM,21.27,21.27,0.00
101002smtd202300241,https://www.doi.org/10.1002/smtd.202300241,2023.0,2,1,LLM,22.93,21.10,-1.83
101038s4159802332697z,https://www.doi.org/10.1038/s41598-023-32697-z,2023.0,5,5,LLM,20.40,20.40,0.00
101039d3ta05069f,https://www.doi.org/10.1039/d3ta05069f,2023.0,5,1,LLM,22.91,20.04,-2.87
101016jheliyon2024e25352,https://www.doi.org/10.1016/j.heliyon.2024.e25352,2024.0,5,1,LLM,20.50,18.70,-1.80
101021acsaem4c02471,https://www.doi.org/10.1021/acsaem.4c02471,2024.0,5,2,LLM,16.00,16.00,0.00


**อ่านยังไง:**
- `n_ours` / `n_nomad` = จำนวนเครื่องที่แต่ละฐานเก็บจากเปเปอร์เดียวกัน
- `pce_max_*` = ประสิทธิภาพสูงสุดของเปเปอร์นั้นที่แต่ละฐานบันทึก — **เปเปอร์เดียวกันควรได้ค่าเท่ากัน**
  ถ้า `pce_diff` ติดลบ แปลว่า NOMAD ไม่ได้เก็บเครื่องที่ดีที่สุดของเปเปอร์ไว้ (การเทียบนี้อยู่ในเปเปอร์เดียวกัน ไม่ใช่การเทียบ PCE ข้ามเปเปอร์)

ตารางถัดไปเรียง PCE ของทุกเครื่องในเปเปอร์ที่ซ้ำ เพื่อดูว่า NOMAD เก็บเครื่องไหนไว้

In [7]:
pce_lists = pd.DataFrame({
    "PCE ทุกเครื่อง (ของเรา)": ours[ours["doi_key"].isin(overlap.index)].groupby("doi_key")["pce"].apply(lambda s: sorted(s.dropna().round(2))),
    "PCE ทุกเครื่อง (NOMAD)": nmf[nmf["doi_key"].isin(overlap.index)].groupby("doi_key")["pce"].apply(lambda s: sorted(s.dropna().round(2))),
}).join(overlap[["doi_nomad", "method"]])
pce_lists

,PCE ทุกเครื่อง (ของเรา),PCE ทุกเครื่อง (NOMAD),doi_nomad,method
doi_key,,,,
101002smtd202200787,"[19.01, 21.27]","[19.21, 21.27]",https://www.doi.org/10.1002/smtd.202200787,LLM
101002smtd202300241,"[21.1, 22.93]",[21.1],https://www.doi.org/10.1002/smtd.202300241,LLM
101007s1085401780949,"[9.1, 9.3, 12.7, 12.8]","[8.3, 9.1, 9.3, 10.8, 11.3, 12.5, 12.6, 12.6, 12.7, 12.8]",10.1007/s10854-017-8094-9,manual
101016jheliyon2024e25352,"[18.2, 18.7, 19.3, 19.4, 20.5]",[18.7],https://www.doi.org/10.1016/j.heliyon.2024.e25352,LLM
101021acsaem4c02471,"[10.34, 13.08, 14.32, 14.35, 16.0]","[13.08, 16.0]",https://www.doi.org/10.1021/acsaem.4c02471,LLM
101038s4159802332697z,"[16.6, 18.1, 18.8, 19.5, 20.4]","[16.6, 18.1, 18.8, 19.5, 20.4]",https://www.doi.org/10.1038/s41598-023-32697-z,LLM
101039d3ta05069f,"[18.23, 20.76, 21.28, 21.68, 22.91]",[20.04],https://www.doi.org/10.1039/d3ta05069f,LLM


**อ่านยังไง** (ณ snapshot นี้):
- **ตรงกันดี 3 เปเปอร์** (`smtd.202200787`, `s41598-023-32697-z`, `acsaem.4c02471`) — ค่าที่ NOMAD มีตรงกับเครื่องของเรา
  (ต่างกันแค่ 19.01 vs 19.21 หนึ่งค่า น่าจะเป็นสแกนคนละทิศ)
- **NOMAD เก็บแค่ 1 จาก 2–5 เครื่อง 3 เปเปอร์** — `smtd.202300241` ได้แค่เครื่องควบคุม (21.1) ไม่มีเครื่องที่ปรับปรุง ·
  `heliyon.2024.e25352` ได้ 18.7 ไม่มีตัวดีสุด 20.5 · `d3ta05069f` ค่า 20.04 **ไม่ตรงกับเครื่องไหนของเราเลย** ต้องเปิดเปเปอร์ดูว่าฝั่งไหนถูก
- **NOMAD เก็บมากกว่าเรา 1 เปเปอร์** (`s10854-017-8094-9`, คนกรอก) — มีทั้งค่า best และ average และชุดที่มีชั้น ZnO ซึ่งทีมเราไม่ได้เก็บ

ทั้งสามเคสที่ NOMAD ได้ไม่ครบเป็นเปเปอร์ที่ LLM กรอก — เมื่อขาดเครื่องที่ปรับปรุงหรือเครื่องควบคุม ก็คิด Δ แบบที่เราใช้ไม่ได้ (ดูข้อ 5.5)

### 4.2 ซ้ำกับ NOMAD ทั้งฐาน (ไม่กรอง)
บางเปเปอร์ของเรามีใน NOMAD แต่ไม่ผ่านเงื่อนไข — ดูว่ามีกี่เปเปอร์ และตกเงื่อนไขข้อไหน

In [8]:
nomad_dois["doi_key"] = nomad_dois["doi"].map(doi_key)
in_nomad = papers_ours.index.intersection(nomad_dois["doi_key"])
outside = in_nomad.difference(papers_nm.index)
print(f"เปเปอร์ของเราที่มีใน NOMAD ทั้งฐาน: {len(in_nomad)} / {len(papers_ours)}")
print(f"  ผ่านเงื่อนไข (ข้อ 4.1): {len(overlap)}   ·   มีใน NOMAD แต่ไม่ผ่านเงื่อนไข: {len(outside)}")
print(f"เปเปอร์ของเราที่ NOMAD ไม่มีเลย: {len(papers_ours) - len(in_nomad)} ({1 - len(in_nomad) / len(papers_ours):.0%})")


def fetch_outside():
    out = []
    for doi_str in nomad_dois.loc[nomad_dois["doi_key"].isin(outside), "doi"]:
        q = {"section_defs.definition_qualified_name:all": [SCHEMA], f"data.ref.DOI_number#{SCHEMA}": doi_str}
        out += download_archives(q)
    return out


why = pd.DataFrame([flatten(a) for a in cached("nomad_overlap_outside_filter.json", fetch_outside)["items"]])
why["ข้อ1 Cs;FA;MA"] = why["a_ions"].eq("Cs; FA; MA")
why["ข้อ2 NiO HTL"] = why["htl_list"].map(lambda h: bool(set(h or []) & {"NiOx", "NiO-c", "NiO-np"}))
why["ข้อ3 pin"] = why["architecture"].eq("pin")
why["doi_key"] = why["doi"].map(doi_key)
ours_comp = (ours[["device_id", "doi_key"]].join(comp[["layer_material", "a_Cs"]], on="device_id")
             .groupby("doi_key").agg(สูตรที่เราบันทึก=("layer_material", lambda s: " · ".join(sorted(set(s.dropna())))),
                                     Cs_ของเรา=("a_Cs", "max")))
why.join(ours_comp, on="doi_key")[["doi", "method", "composition", "สูตรที่เราบันทึก", "Cs_ของเรา", "htl_stack", "pce",
                                    "ข้อ1 Cs;FA;MA", "ข้อ2 NiO HTL", "ข้อ3 pin"]]

เปเปอร์ของเราที่มีใน NOMAD ทั้งฐาน: 9 / 43
  ผ่านเงื่อนไข (ข้อ 4.1): 7   ·   มีใน NOMAD แต่ไม่ผ่านเงื่อนไข: 2
เปเปอร์ของเราที่ NOMAD ไม่มีเลย: 34 (79%)


,doi,method,composition,สูตรที่เราบันทึก,Cs_ของเรา,htl_stack,pce,ข้อ1 Cs;FA;MA,ข้อ2 NiO HTL,ข้อ3 pin
0,https://www.doi.org/10.1002/solr.202400534,LLM,FA0.85MA0.15PbBr0.45I2.55,Cs0.05(FA0.83MA0.12)0.95Pb(I0.83Br0.17)3 · Cs0.05(FA0.83MA0.12)0.9...,0.053,NiOx@Me-2PACz,20.37,False,False,True
1,https://www.doi.org/10.1002/solr.202400534,LLM,FA0.85MA0.15PbBr0.45I2.55,Cs0.05(FA0.83MA0.12)0.95Pb(I0.83Br0.17)3 · Cs0.05(FA0.83MA0.12)0.9...,0.053,NiOx@Me-2PACz,24.21,False,False,True
2,https://www.doi.org/10.1039/d2tc04215k,LLM,FA0.85MA0.15PbBr0.45I2.55,Cs0.05MA0.10FA0.85Pb(I0.97Br0.03)3,0.050,NiOx,18.60,False,True,True
3,https://www.doi.org/10.1039/d2tc04215k,LLM,FA0.85MA0.15PbBr0.45I2.55,Cs0.05MA0.10FA0.85Pb(I0.97Br0.03)3,0.050,NiOx,16.60,False,True,True
4,https://www.doi.org/10.1039/d2tc04215k,LLM,FA0.85MA0.15PbBr0.45I2.55,Cs0.05MA0.10FA0.85Pb(I0.97Br0.03)3,0.050,NiOx,17.00,False,True,True
5,https://www.doi.org/10.1039/d2tc04215k,LLM,FA0.85MA0.15PbBr0.45I2.55,Cs0.05MA0.10FA0.85Pb(I0.97Br0.03)3,0.050,PTAA,20.60,False,False,True


**อ่านยังไง:** คอลัมน์ `ข้อ1–ข้อ3` = False คือเงื่อนไขที่ทำให้เครื่องนั้นหลุด
- ทั้งสองเปเปอร์หลุดข้อ 1 เพราะ NOMAD (กรอกโดย LLM) บันทึกสูตรเป็น `FA0.85MA0.15…` **ไม่มี Cs** แต่ทีมเราอ่านจากเปเปอร์แล้วบันทึกว่า **มี Cs**
  (`Cs_ของเรา` > 0) — ถ้าเราอ่านถูก แปลว่า LLM ใส่สูตรผิด (สูตร FA0.85MA0.15 เป็นสูตรยอดนิยม อาจถูกเดาแทนสูตรจริง) ควรเปิดเปเปอร์ยืนยัน
- `solr.202400534` หลุดข้อ 2 ด้วย เพราะ NOMAD เขียน HTL เป็น `NiOx@Me-2PACz` (ชื่อไม่มาตรฐาน) แทนที่จะแยกเป็น `NiOx | Me-2PACz`
- แถวที่ HTL เป็น `PTAA` คือเครื่องอ้างอิงในเปเปอร์ที่ไม่ใช้ NiOx ทีมเราไม่ได้เก็บ จึงหลุดถูกต้อง

**สรุปข้อ 1:** ซ้ำ **7 เปเปอร์** ตามเงื่อนไขที่กำหนด · ถ้า NOMAD บันทึกสูตรถูก จะเป็น **9 เปเปอร์** ·
ไม่ว่าแบบไหน เปเปอร์ของเรา **34 จาก 43 (79%) ไม่มีใน NOMAD เลย**

## 5. งานที่ 2 — จุดแข็งของฐานข้อมูลเรา

เทียบ **ฐานข้อมูลเราทั้งหมด** (ทุกเครื่องเป็น NiOx + pin และ 124/134 เครื่องเป็น Cs/FA/MA)
กับ NOMAD ชุดที่กรองแล้ว โดย **แยก manual ออกจาก LLM ในทุกตาราง**

| คอลัมน์ | คืออะไร | ใช้เทียบยังไง |
|---|---|---|
| ของเรา | ทีมอ่านเปเปอร์แล้วกรอกเอง | — |
| **NOMAD manual** | คนกรอกเหมือนกัน (ชุดดั้งเดิม ถึงปี 2021) | **ตัวเทียบหลัก** — งานมือทั้งคู่ ความต่างจึงมาจาก *สิ่งที่เลือกเก็บ* ไม่ใช่วิธีผลิตข้อมูล |
| NOMAD LLM | โมเดลภาษาอ่านเปเปอร์แล้วกรอก (ปี 2021 ขึ้นไป) | ดูแยก — ความต่างปนทั้งการออกแบบ schema และความแม่นของการสกัด |

**ทำไมต้องแยก** ถ้ารวมสองชุดแล้วสรุปว่า "ของเราครอบคลุมกว่า" จะตอบไม่ได้ว่าชนะเพราะ
*เราออกแบบให้เก็บมากกว่า* หรือเพราะ *อีกฝั่งเป็นข้อมูล LLM ที่ยังสกัดได้ไม่ครบ* —
เป็นคนละข้อสรุป และนำไปสู่คนละการตัดสินใจว่าจะเอา NOMAD มาใช้ยังไง

ทุกตารางรายงานเป็น "จำนวนเครื่องที่มีข้อมูลช่องนั้น / ทั้งหมดของชุดนั้น" ·
ช่องที่ schema ของ NOMAD **ไม่มีที่ให้เก็บเลย** แสดงเป็น `ไม่มีช่อง` (ตรวจจากโค้ด schema ในไลบรารี ไม่ใช่แค่ว่าง)

In [9]:
# ตั้งแต่นี้ไป NOMAD ถูกแยกเป็นสองชุดเสมอ ไม่รวมเป็นก้อนเดียว
#   manual = ชุดดั้งเดิมที่คนอ่านเปเปอร์แล้วกรอก -- เทียบกับฐานเราได้ตรง ๆ เพราะเป็นงานมือเหมือนกัน
#   LLM    = ชุดที่โมเดลภาษากรอกอัตโนมัติ -- คนละวิธีผลิตข้อมูล ต้องแยกดู
nmf_man = nmf[nmf["method"] == "manual"].copy()
nmf_llm = nmf[nmf["method"] == "LLM"].copy()

N_OURS, N_NM = len(ours), len(nmf)
N_MAN, N_LLM = len(nmf_man), len(nmf_llm)
COL_MAN, COL_LLM = "NOMAD manual (คนกรอก)", "NOMAD LLM"


def pct(n, d):
    return f"{n}/{d} ({n / d:.0%})" if d else "—"


def nm_cov(col, df=None):
    d = nmf if df is None else df
    return pct(int(d[col].map(known).sum()), len(d))


def cov_man(col):
    return nm_cov(col, nmf_man)


def cov_llm(col):
    return nm_cov(col, nmf_llm)


def ours_cov(mask_or_ids):
    ids = set(mask_or_ids)
    return pct(len(ids & set(ours["device_id"])), N_OURS)


def yr_range(s):
    return f"{int(s.min())}–{int(s.max())}"


pd.DataFrame({
    "เครื่อง": [N_OURS, N_MAN, N_LLM],
    "เปเปอร์": [len(papers_ours), nmf_man["doi_key"].nunique(), nmf_llm["doi_key"].nunique()],
    "ช่วงปีตีพิมพ์": [yr_range(papers_ours["year"]), yr_range(nmf_man["year"]), yr_range(nmf_llm["year"])],
    "ซ้ำกับฐานเรา (เปเปอร์)": ["—", len(ov_man), len(ov_llm)],
}, index=["ฐานข้อมูลเรา", COL_MAN, COL_LLM])

,เครื่อง,เปเปอร์,ช่วงปีตีพิมพ์,ซ้ำกับฐานเรา (เปเปอร์)
ฐานข้อมูลเรา,134,43,2017–2025,—
NOMAD manual (คนกรอก),206,47,2017–2021,1
NOMAD LLM,183,74,2021–2026,6


### 5.1 ความใหม่ — เปเปอร์ของเราใหม่กว่า และส่วนใหญ่ NOMAD ไม่มี

In [10]:
BINS = [0, 2020, 2021, 2022, 2023, 2024, 2025, 2100]
LABELS = ["≤2020", "2021", "2022", "2023", "2024", "2025", "2026"]
yr_ours = pd.cut(papers_ours["year"], BINS, labels=LABELS).value_counts().reindex(LABELS)
nm_papers_year = nmf.groupby("doi_key")["year"].first()
yr_nm = pd.cut(nm_papers_year, BINS, labels=LABELS).value_counts().reindex(LABELS)
yr_llm = pd.cut(nmf[nmf.method == "LLM"].groupby("doi_key")["year"].first(), BINS, labels=LABELS).value_counts().reindex(LABELS)

recency = pd.DataFrame({"เปเปอร์ของเรา": yr_ours, "เปเปอร์ NOMAD (กรองแล้ว)": yr_nm, "  └ ในนั้นกรอกโดย LLM": yr_llm}).fillna(0).astype(int)
recency.loc["รวม"] = recency.sum()
recency.loc["ค่ากลางปี (median)"] = [int(papers_ours["year"].median()), int(nm_papers_year.median()),
                                     int(nmf[nmf.method == "LLM"].groupby("doi_key")["year"].first().median())]
recency

,เปเปอร์ของเรา,เปเปอร์ NOMAD (กรองแล้ว),└ ในนั้นกรอกโดย LLM
year,,,
≤2020,2,46,0
2021,6,3,2
2022,8,12,12
2023,8,15,15
2024,7,23,23
2025,12,18,18
2026,0,4,4
รวม,43,121,74
ค่ากลางปี (median),2023,2022,2024


**อ่านยังไง:** แถว = ปีตีพิมพ์ของเปเปอร์ · NOMAD ชุดดั้งเดิม (คนกรอก) หยุดราวปี 2020 ส่วนปีหลังจากนั้นมาจาก LLM เกือบทั้งหมด
ดังนั้นสำหรับเปเปอร์ NiOx ช่วง 2021–2025 ซึ่งเป็นช่วงที่ SAM และเทคนิคเคลือบผิวบูมที่สุด ฐานของเราเป็นข้อมูลที่ **คนอ่านและตรวจเอง**
ขณะที่ฝั่ง NOMAD เป็นข้อมูลที่ LLM ดึงออกมา — และข้อ 4.2 แสดงว่าเปเปอร์ของเราส่วนใหญ่ NOMAD ยังไม่มีเลย

### 5.2 ข้อมูลความเสถียรพร้อมสภาวะทดสอบ

งานของเราเน้น "ทนแค่ไหน" ด้วย จึงเก็บผลทดสอบความเสถียรพร้อมสภาวะทดสอบ (อุณหภูมิ ความชื้น แสง บรรยากาศ ระยะเวลา)

In [11]:
stab = pd.read_csv(P / "stability.csv")
s_num = stab.loc[stab["retention_pct_parsed"].notna() | stab["t80_hr_parsed"].notna(), "device_id"]
s_time = stab.loc[pd.to_numeric(stab["time_hr"], errors="coerce").notna(), "device_id"]
s_temp = stab.loc[stab["temperature_c_parsed"].notna(), "device_id"]
s_rh = stab.loc[stab["humidity_pct_parsed"].notna() | stab["humidity_pct_min"].notna(), "device_id"]
s_env = stab.loc[stab["environment"].map(known), "device_id"]

for _d in (nmf, nmf_man, nmf_llm):
    _d["stab_num"] = _d["t80_h"].notna() | _d["pce_end"].notna()
n_tests = stab.groupby("device_id").size()

NO_SLOT = "ไม่มีช่อง (schema เก็บได้เครื่องละ 1 การทดสอบ)"


def stab_col(d):
    n = len(d)
    return [pct(int(d["stab_num"].sum()), n), nm_cov("stab_time_h", d), nm_cov("stab_temp", d),
            nm_cov("stab_rh", d),
            pct(int((d["stab_light"].map(known) | d["stab_atm"].map(known)).sum()), n), NO_SLOT]


stab_tbl = pd.DataFrame({
    "ของเรา": [ours_cov(s_num), ours_cov(s_time), ours_cov(s_temp), ours_cov(s_rh), ours_cov(s_env),
               ours_cov(n_tests[n_tests >= 2].index)],
    COL_MAN: stab_col(nmf_man),
    COL_LLM: stab_col(nmf_llm),
}, index=["มีตัวเลขความเสถียร (retention / T80 / PCE ปลายการทดสอบ)", "ระยะเวลาทดสอบ", "อุณหภูมิทดสอบ",
          "ความชื้นทดสอบ", "สภาวะแสง/บรรยากาศ", "ทดสอบ ≥ 2 สภาวะในเครื่องเดียว (เช่น แสง + ความร้อน)"])
stab_tbl

,ของเรา,NOMAD manual (คนกรอก),NOMAD LLM
มีตัวเลขความเสถียร (retention / T80 / PCE ปลายการทดสอบ),76/134 (57%),52/206 (25%),23/183 (13%)
ระยะเวลาทดสอบ,80/134 (60%),52/206 (25%),23/183 (13%)
อุณหภูมิทดสอบ,65/134 (49%),51/206 (25%),12/183 (7%)
ความชื้นทดสอบ,52/134 (39%),4/206 (2%),0/183 (0%)
สภาวะแสง/บรรยากาศ,86/134 (64%),52/206 (25%),10/183 (5%)
ทดสอบ ≥ 2 สภาวะในเครื่องเดียว (เช่น แสง + ความร้อน),34/134 (25%),ไม่มีช่อง (schema เก็บได้เครื่องละ 1 การทดสอบ),ไม่มีช่อง (schema เก็บได้เครื่องละ 1 การทดสอบ)


**อ่านยังไง:** ฝั่ง NOMAD นับ "มีตัวเลข" แบบใจกว้าง (มี T80 หรือมีค่าปลายการทดสอบอย่างเดียวก็นับ)

- เทียบกับ **manual** (งานมือเหมือนกัน) เรายังนำทุกช่อง — ช่องที่ห่างที่สุดคือ **ความชื้น 39% vs 2%**
  ซึ่งสำคัญมากเพราะ perovskite เสื่อมจากน้ำ · ส่วนอุณหภูมิและระยะเวลาห่างราวสองเท่า
- **LLM ต่ำกว่า manual ทุกช่อง** และต่ำกว่ามากในอุณหภูมิ (7% vs 25%) กับสภาวะแสง/บรรยากาศ (5% vs 25%)
  → ยิ่งเปเปอร์ใหม่ NOMAD ยิ่งมีสภาวะทดสอบน้อยลง
- แถวสุดท้ายเป็นข้อจำกัดของ **schema** ไม่ใช่ของผู้กรอก: เครื่องเดียวของเราถูกทดสอบได้หลายสภาวะ
  (stability.csv 134 แถว จาก 86 เครื่อง) ทำให้เทียบได้ว่าการปรับปรุงช่วยทนแสง ทนร้อน หรือทนชื้น —
  NOMAD เก็บได้ชุดเดียวต่อเครื่องทั้งสองชุด

In [12]:
by_method = nmf.groupby("method").agg(
    เครื่อง=("entry_id", "size"),
    มีตัวเลขความเสถียร=("stab_num", "sum"),
    มีระยะเวลาทดสอบ=("stab_time_h", lambda s: int(s.notna().sum())),
    มีอุณหภูมิทดสอบ=("stab_temp", lambda s: int(s.map(known).sum())),
)
print("NOMAD แยกตามผู้กรอก:")
print(by_method.to_string(), "\n")

e = nmf[nmf["pce_end"].notna()].copy()
e["ความหมายที่ดูจากค่า"] = (((e["pce_end"] > 30) | (e["pce_end"] > e["pce_init"]))
                            .map({True: "% คงเหลือ (retention)", False: "PCE จริง (%)"}))
pd.crosstab(e["method"], e["ความหมายที่ดูจากค่า"], margins=True, margins_name="รวม")

NOMAD แยกตามผู้กรอก:
        เครื่อง  มีตัวเลขความเสถียร  มีระยะเวลาทดสอบ  มีอุณหภูมิทดสอบ
method                                                               
LLM         183                  23               23               12
manual      206                  52               52               51 



ความหมายที่ดูจากค่า,% คงเหลือ (retention),PCE จริง (%),รวม
method,,,
LLM,0,23,23
manual,48,4,52
รวม,48,27,75


**อ่านยังไง:**
- ตารางแรก: ชุดที่ LLM กรอก (เปเปอร์ปีหลัง ๆ) มีตัวเลขความเสถียร 13% ของเครื่อง ครึ่งหนึ่งของชุดที่คนกรอก (25%)
  และบอกอุณหภูมิทดสอบแค่ราวครึ่งของที่มีตัวเลข — ยิ่งเปเปอร์ใหม่ NOMAD ยิ่งมีข้อมูลความเสถียรน้อยลง
- ตารางที่สอง: ช่อง `PCE end of experiment` ตาม schema ต้องเป็น PCE หน่วย % แต่ค่าเกิน 30% หรือเกิน PCE ตั้งต้นเป็นไปไม่ได้ทางฟิสิกส์
  (สถิติโลกของ perovskite ~27%) จึงเป็น "% คงเหลือ" — **ชุดคนกรองเก็บเป็น % คงเหลือ (48/52) ส่วนชุด LLM เก็บเป็น PCE จริง (23/23)**
  ช่องเดียวกันสองความหมาย ถ้าเอาไปใช้ตรง ๆ จะผิดโดยไม่รู้ตัว (และทำให้เงื่อนไข 4 แบบ "initial + end" นับปนกันด้วย)
- ของเราแยก `retention_pct` กับ `initial_pce/final_pce` คนละคอลัมน์ และมีธง `*_censor`
  บอกว่าค่าไหนเป็น "อย่างน้อย/ไม่ถึง" (เช่น `<50` = เสื่อมจนต่ำกว่า 50%)

### 5.3 พารามิเตอร์กระบวนการผลิต (knob ที่เอากลับไปทำในแล็บได้)

เป้าหมายโปรเจกต์คือหา process ที่ทำซ้ำในแล็บได้ จึงต้องมีตัวเลขของขั้นตอนผลิต ไม่ใช่แค่ชื่อวิธี

In [13]:
proc = pd.read_csv(P / "process.csv", low_memory=False)
mats = pd.read_csv(P / "materials.csv")
lay = pd.read_csv(P / "layers.csv")
ven = pd.read_csv(P / "vendors.csv")


def ids(df, mask):
    return df.loc[mask, "device_id"]


ROWS = [
    ("HTL: อุณหภูมิอบ", ids(tt, tt["htl_anneal_C"].notna()), "htl_anneal_T"),
    ("HTL: เวลาอบ", ids(proc, (proc["layer_id"] == 2) & proc["Annealing__time_s"].notna()), "htl_anneal_t"),
    ("HTL: ความหนา", ids(lay, (lay["layer_id"] == 2) & lay["thickness_nm_parsed"].notna()), "htl_thick"),
    ("Perovskite: ความเข้มข้น/ปริมาณสารตั้งต้น", ids(mats, (mats["layer_id"] == 4) & mats["amount_parsed"].notna()), "psk_conc"),
    ("Perovskite: ความเร็วรอบ spin (rpm)", ids(tt, tt["psk_spin_rpm"].notna()), None),
    ("Perovskite: ปริมาตร antisolvent", ids(tt, tt["antisolvent_vol_uL"].notna()), "quench_vol"),
    ("Perovskite: จังหวะหยด antisolvent (วินาที)", ids(tt, tt["antisolvent_timing_s"].notna()), None),
    ("Perovskite: อุณหภูมิอบ", ids(tt, tt["psk_anneal_C"].notna()), "psk_anneal_T"),
    ("Perovskite: เวลาอบ", ids(tt, tt["psk_anneal_time_s"].notna()), "psk_anneal_t"),
    ("Perovskite: ความหนา", ids(lay, (lay["layer_id"] == 4) & lay["thickness_nm_parsed"].notna()), "psk_thick"),
    ("สารเคมี: ผู้ผลิต (vendor)", ids(ven, ven["vendor"].notna()), "supplier"),
    ("สารเคมี: ความบริสุทธิ์ (purity)", ids(ven, ven["purity_pct"].notna()), "purity"),
]
proc_tbl = pd.DataFrame(
    {"ของเรา": [ours_cov(o) for _, o, _ in ROWS],
     COL_MAN: [cov_man(c) if c else "ไม่มีช่อง" for _, _, c in ROWS],
     COL_LLM: [cov_llm(c) if c else "ไม่มีช่อง" for _, _, c in ROWS]},
    index=[r for r, _, _ in ROWS])
proc_tbl

,ของเรา,NOMAD manual (คนกรอก),NOMAD LLM
HTL: อุณหภูมิอบ,114/134 (85%),26/206 (13%),36/183 (20%)
HTL: เวลาอบ,107/134 (80%),17/206 (8%),35/183 (19%)
HTL: ความหนา,52/134 (39%),53/206 (26%),14/183 (8%)
Perovskite: ความเข้มข้น/ปริมาณสารตั้งต้น,117/134 (87%),13/206 (6%),63/183 (34%)
Perovskite: ความเร็วรอบ spin (rpm),119/134 (89%),ไม่มีช่อง,ไม่มีช่อง
Perovskite: ปริมาตร antisolvent,98/134 (73%),26/206 (13%),0/183 (0%)
Perovskite: จังหวะหยด antisolvent (วินาที),96/134 (72%),ไม่มีช่อง,ไม่มีช่อง
Perovskite: อุณหภูมิอบ,128/134 (96%),184/206 (89%),58/183 (32%)
Perovskite: เวลาอบ,128/134 (96%),176/206 (85%),58/183 (32%)
Perovskite: ความหนา,40/134 (30%),48/206 (23%),25/183 (14%)


**อ่านยังไง:** แต่ละแถวคือ "ปุ่มปรับ" หนึ่งตัวของการผลิต · อ่านคอลัมน์ **manual** เป็นหลัก

- `ไม่มีช่อง` = schema ของ NOMAD **ไม่ได้ออกแบบให้เก็บเลย** — ความเร็วรอบ spin และจังหวะหยด antisolvent
  เป็น knob หลักที่คุมคุณภาพฟิล์ม perovskite และไม่มีที่เก็บทั้งสองชุด
- **ฝั่ง HTL (NiOx) ซึ่งเป็นหัวใจของโปรเจกต์**: เรามีอุณหภูมิ/เวลาอบ 80–85% ส่วน manual 8–13%
  และ LLM 19–20% — ทั้งสองชุดของ NOMAD ห่างจากเรามาก
- **จุดที่ต้องไม่อ้างเกินจริง**: การอบ perovskite ชุด **manual ได้ 85–89% ใกล้เคียงของเรา (96%)**
  ไม่ใช่จุดแข็งของเรา · ตัวเลขรวมที่เคยดูเหมือนเราชนะขาดเกิดจาก LLM (32%) ถ่วงค่าเฉลี่ยลง
- **สองชุดเก่งคนละช่อง**: manual ดีกว่าที่ความหนา HTL (26% vs 8%) และปริมาตร antisolvent (13% vs 0%)
  ส่วน LLM ดีกว่าที่ความเข้มข้นสารตั้งต้น (34% vs 6%) และการอบ HTL — สกัดได้ไม่เหมือนกัน
- ข้อมูลผู้ผลิต/ความบริสุทธิ์สารเคมีแทบไม่มีทั้งสองชุด (manual 9%/0% · LLM 0%/0%) ขณะที่เรามี 95%/68%
- ของเราเก็บเป็นขั้นตอนทีละสเต็ปพร้อมหน่วย (`process.csv` ~2 พันแถว) จึงทำซ้ำในแล็บได้ทั้งลำดับ ไม่ใช่แค่ค่าสรุป

### 5.4 ผลวัดคุณสมบัติฟิล์ม/รอยต่อ (ตัวกลางที่อธิบาย "ทำไม" ประสิทธิภาพเปลี่ยน)

การวิเคราะห์กลไกของเราใช้ตัวกลางเหล่านี้ (เช่น trap density, ขนาดเกรน, AFM rms, มุมสัมผัส) — ไม่รวม VTFL ตามที่ตกลงไว้

In [14]:
props = pd.read_csv(P / "properties.csv")


def prop_ids(pattern):
    m = props["base_property"].str.contains(pattern, case=False, na=False, regex=True) & props["value_parsed"].notna()
    return props.loc[m, "device_id"]


def hi_cov(d):
    return pct(int((d["rev_pce"].notna() & d["fwd_pce"].notna()).sum()), len(d))


NONE_ = "ไม่มีช่อง"
CHAR = [
    ("Trap density (ความหนาแน่นจุดบกพร่อง)", ids(tt, tt["trap_density"].notna()), NONE_, NONE_),
    ("ขนาดเกรน (grain size) *", ids(tt, tt["grain_size"].notna()), NONE_, NONE_),
    ("ความขรุขระผิว (AFM rms)", ids(tt, tt["afm_rms"].notna()),
     f"ไม่มีช่องของฟิล์ม (substrate: {cov_man('sub_rms')})",
     f"ไม่มีช่องของฟิล์ม (substrate: {cov_llm('sub_rms')})"),
    ("มุมสัมผัสน้ำ (contact angle)", ids(tt, tt["contact_angle"].notna()), NONE_, NONE_),
    ("ระดับพลังงาน: work function / VBM", ids(tt, tt["work_function"].notna() | tt["vbm"].notna()), NONE_, NONE_),
    ("ความต้านทาน Rs / Rrec", ids(tt, tt["rs"].notna() | tt["rrec"].notna()), NONE_, NONE_),
    ("อายุพาหะ PL lifetime / TRPL", prop_ids(r"PL Lifetime|TRPL"),
     f"ไม่มีช่อง (มีแค่ยอด PL: {cov_man('pl_max')})", f"ไม่มีช่อง (มีแค่ยอด PL: {cov_llm('pl_max')})"),
    ("Hysteresis (HI) / สแกนสองทิศ", ids(tt, tt["hi"].notna()),
     f"{hi_cov(nmf_man)} มี forward+reverse", f"{hi_cov(nmf_llm)} มี forward+reverse"),
    ("Jsc จาก EQE", prop_ids(r"EQE"), cov_man("eqe_jsc"), cov_llm("eqe_jsc")),
]
char_tbl = pd.DataFrame({"ของเรา": [ours_cov(o) for _, o, _, _ in CHAR],
                         COL_MAN: [m for _, _, m, _ in CHAR],
                         COL_LLM: [l for _, _, _, l in CHAR]}, index=[r for r, _, _, _ in CHAR])
char_tbl

,ของเรา,NOMAD manual (คนกรอก),NOMAD LLM
Trap density (ความหนาแน่นจุดบกพร่อง),57/134 (43%),ไม่มีช่อง,ไม่มีช่อง
ขนาดเกรน (grain size) *,24/134 (18%),ไม่มีช่อง,ไม่มีช่อง
ความขรุขระผิว (AFM rms),43/134 (32%),ไม่มีช่องของฟิล์ม (substrate: 0/206 (0%)),ไม่มีช่องของฟิล์ม (substrate: 0/183 (0%))
มุมสัมผัสน้ำ (contact angle),41/134 (31%),ไม่มีช่อง,ไม่มีช่อง
ระดับพลังงาน: work function / VBM,61/134 (46%),ไม่มีช่อง,ไม่มีช่อง
ความต้านทาน Rs / Rrec,43/134 (32%),ไม่มีช่อง,ไม่มีช่อง
อายุพาหะ PL lifetime / TRPL,14/134 (10%),ไม่มีช่อง (มีแค่ยอด PL: 28/206 (14%)),ไม่มีช่อง (มีแค่ยอด PL: 0/183 (0%))
Hysteresis (HI) / สแกนสองทิศ,26/134 (19%),66/206 (32%) มี forward+reverse,0/183 (0%) มี forward+reverse
Jsc จาก EQE,17/134 (13%),36/206 (17%),0/183 (0%)


**อ่านยังไง:** ของเราเก็บผลวัดฟิล์มและรอยต่อหลายชนิด ซึ่งเป็นตัวที่ใช้อธิบายกลไก (เช่น "เคลือบผิว → trap ลด → Voc ขึ้น")
ส่วน NOMAD ไม่มีช่องเก็บเกือบทั้งหมด → **ใช้ NOMAD ตอบได้แค่ "อะไรให้ PCE สูง" แต่ตอบ "ทำไม" ไม่ได้**
สองแถวล่าง (HI, EQE) ทั้งสองฝั่งมีช่องเก็บและมีข้อมูลพอ ๆ กัน — ไม่นับเป็นจุดแข็งของฝั่งไหน

\* ขนาดเกรนของเรายังปน Top/Bottom/ไม่ระบุผิว — ตามมติที่ประชุม 28 ก.ย. วิเคราะห์ได้เฉพาะแถวที่ยืนยันว่าเป็น Top
(ดู `notebooks/deep_info/bottom_top_grain.ipynb`) ตัวเลขนี้คือ "มีการวัด" ไม่ใช่ "พร้อมใช้ทั้งหมด"

### 5.5 คู่ "เครื่องควบคุม vs เครื่องที่ปรับปรุง" ในเปเปอร์เดียวกัน

การเทียบ PCE ข้ามเปเปอร์ใช้ไม่ได้ (83% ของความต่างมาจาก "เปเปอร์ไหนทำ") วิธีที่ถูกคือ **Δ จากเครื่องควบคุมในเปเปอร์เดียวกัน**
ซึ่งต้องรู้ว่าเครื่องไหนเป็นตัวควบคุม · ใช้กติกาเดียวกันทั้งสองฝั่ง: เครื่องควบคุม = NiO เปล่า ๆ ไม่มีชั้นเคลือบ/สารเติมใน HTL
(ของเรา: `htl_mod_kind` ว่าง · NOMAD: `htl.stack_sequence` มีชั้นเดียวและไม่มี `htl.additives_compounds`)

In [15]:
o = ours.dropna(subset=["pce"]).assign(treated=lambda d: d["htl_mod_kind"].notna())
pair_ours = o.groupby("doi_key")["treated"].agg(lambda s: s.nunique() == 2)

BARE = {"NiOx", "NiO-c", "NiO-np"}


def pair_stats(d):
    """[เปเปอร์ที่มีคู่ควบคุม-ปรับปรุง, เปเปอร์ที่มี >= 2 เครื่อง] ภายในชุดที่ส่งเข้ามา"""
    x = d.dropna(subset=["pce"]).assign(
        treated=lambda t: ~(t["htl_stack"].isin(BARE) & ~t["htl_additive"].map(known)))
    pair = x.groupby("doi_key")["treated"].agg(lambda s: s.nunique() == 2)
    multi = x.groupby("doi_key").size()
    return [pct(int(pair.sum()), len(pair)), pct(int((multi >= 2).sum()), len(multi))]


pair_tbl = pd.DataFrame({
    "ของเรา": [pct(int(pair_ours.sum()), len(pair_ours)),
               pct(int((o.groupby("doi_key").size() >= 2).sum()), len(pair_ours))],
    COL_MAN: pair_stats(nmf_man),
    COL_LLM: pair_stats(nmf_llm),
}, index=["เปเปอร์ที่มีทั้งเครื่องควบคุมและเครื่องปรับปรุง HTL", "เปเปอร์ที่มี ≥ 2 เครื่อง"])
pair_tbl

,ของเรา,NOMAD manual (คนกรอก),NOMAD LLM
เปเปอร์ที่มีทั้งเครื่องควบคุมและเครื่องปรับปรุง HTL,29/43 (67%),12/47 (26%),1/74 (1%)
เปเปอร์ที่มี ≥ 2 เครื่อง,42/43 (98%),44/47 (94%),48/74 (65%)


**อ่านยังไง:** แถวแรกคือ "เปเปอร์ที่เอาไปคิด Δ ได้ทันที"

ฐานของเราออกแบบมาให้เก็บเครื่องควบคุมคู่กับเครื่องที่ปรับปรุงตั้งแต่ต้น (67%) เพราะคำถามวิจัยคือ
"ปรับ NiOx แล้วดีขึ้นเท่าไร" · ฝั่ง **manual ได้ 26%** และ **LLM ได้ 1%** — ช่องว่างระหว่างสองชุดนี้
กว้างกว่าช่องว่างระหว่างเรากับ manual เสียอีก

สาเหตุต่างกัน: manual มีเครื่องหลายตัวต่อเปเปอร์ (94%) แต่มักเป็นสูตรหรือความหนาต่าง ๆ ไม่ใช่คู่
"NiO เปล่า vs NiO ที่ปรับ" · ส่วน LLM มักดึงมาได้เปเปอร์ละเครื่องเดียว (มี ≥2 เครื่องแค่ 65%)
และเมื่อดึงได้ไม่ครบคู่ ก็คิด Δ ไม่ได้เลย — ตรงกับที่เห็นในข้อ 4.1 ว่า LLM เก็บแค่ 1 จาก 2–5 เครื่องของเปเปอร์

ข้อจำกัด: กติกานี้ดูเฉพาะการปรับที่ HTL ถ้าเปเปอร์ไหนปรับที่ชั้นอื่น จะไม่นับเป็นคู่ทั้งสองฝั่ง

### 5.6 ความสอดคล้องทางฟิสิกส์: PCE ≈ Voc × Jsc × FF

ภายใต้แสง 1 ซัน (100 mW/cm²) ประสิทธิภาพต้องเท่ากับผลคูณสามตัวนี้ — ถ้าต่างเกิน 5% แปลว่ามีตัวเลขสักตัวกรอกผิดหรือมาจากคนละการวัด
pipeline ของเราตรวจข้อนี้ทุกเครื่องและบันทึกเหตุผลของเคสที่ยอมให้ต่างไว้ใน `pipeline/overrides/pce_mismatch_known.csv`

In [16]:
def mismatch(pce, voc, jsc, ff_frac):
    calc = voc * jsc * ff_frac
    ok = pce.notna() & calc.notna()
    rel = ((calc - pce).abs() / pce)[ok]
    return int(ok.sum()), int((rel > 0.05).sum())


def nm_mismatch(d):
    x = d[d["illum"].between(950, 1050) | d["illum"].isna()]
    return mismatch(x["pce"], x["voc"], x["jsc"], x["ff"])


rows = [mismatch(tt["pce"], tt["voc"], tt["jsc"], tt["ff"] / 100),
        nm_mismatch(nmf_man), nm_mismatch(nmf_llm)]
pd.DataFrame({"เครื่องที่มีครบ 4 ค่า": [n for n, _ in rows],
              "ต่างเกิน 5%": [b for _, b in rows],
              "สัดส่วนที่ไม่สอดคล้อง": [f"{b / n:.1%}" if n else "—" for n, b in rows]},
             index=["ของเรา", COL_MAN + " (1 ซัน)", COL_LLM + " (1 ซัน)"])

,เครื่องที่มีครบ 4 ค่า,ต่างเกิน 5%,สัดส่วนที่ไม่สอดคล้อง
ของเรา,129,1,0.8%
NOMAD manual (คนกรอก) (1 ซัน),192,16,8.3%
NOMAD LLM (1 ซัน),136,0,0.0%


**อ่านยังไง:** สัดส่วนยิ่งต่ำยิ่งเชื่อได้ · ข้อควรระวังฝั่ง NOMAD: ช่อง Efficiency อาจเป็นค่า "stabilised" (วัดแบบคงที่) ขณะที่ Voc/Jsc/FF มาจากการสแกน
ความต่างบางส่วนจึงอาจถูกต้องตามจริง — แต่ฐานข้อมูลไม่บอกว่าเคสไหนเป็นแบบนั้น ขณะที่ของเรามีบันทึกเหตุผลรายเครื่อง

### 5.7 เพื่อความยุติธรรม — ด้านที่ NOMAD ดีกว่า

In [17]:
isos_ours = stab["environment"].str.contains("ISOS", case=False, na=False)


def isos_nm(d):
    s = d[d["stab_num"]]
    return f"{int(s['stab_protocol'].map(known).sum())}/{len(s)} เครื่องที่มีผล มีช่อง ISOS แยก"


pd.DataFrame({
    "ของเรา": [f"{len(papers_ours)} เปเปอร์ · {N_OURS} เครื่อง",
               f"{stab.loc[isos_ours, 'device_id'].nunique()}/{s_num.nunique()} เครื่องที่มีผล ระบุรหัส ISOS (ในข้อความ environment)",
               "ไฟล์ Excel ภายในทีม"],
    COL_MAN: [f"{nmf_man['doi_key'].nunique()} เปเปอร์ · {N_MAN} เครื่อง", isos_nm(nmf_man),
              f"สาธารณะ ค้นผ่าน API ได้ · ปิดรับแล้ว (ถึงปี {int(nmf_man['year'].max())})"],
    COL_LLM: [f"{nmf_llm['doi_key'].nunique()} เปเปอร์ · {N_LLM} เครื่อง", isos_nm(nmf_llm),
              f"สาธารณะ · ยังโตต่อ ({int((nm_papers_year >= 2024).sum())} เปเปอร์ปี 2024 ขึ้นไป)"],
}, index=["ขนาด", "โปรโตคอลทดสอบความเสถียรแบบมาตรฐาน (ISOS)", "การเข้าถึง"])

,ของเรา,NOMAD manual (คนกรอก),NOMAD LLM
ขนาด,43 เปเปอร์ · 134 เครื่อง,47 เปเปอร์ · 206 เครื่อง,74 เปเปอร์ · 183 เครื่อง
โปรโตคอลทดสอบความเสถียรแบบมาตรฐาน (ISOS),17/76 เครื่องที่มีผล ระบุรหัส ISOS (ในข้อความ environment),52/52 เครื่องที่มีผล มีช่อง ISOS แยก,0/23 เครื่องที่มีผล มีช่อง ISOS แยก
การเข้าถึง,ไฟล์ Excel ภายในทีม,สาธารณะ ค้นผ่าน API ได้ · ปิดรับแล้ว (ถึงปี 2021),สาธารณะ · ยังโตต่อ (45 เปเปอร์ปี 2024 ขึ้นไป)


**อ่านยังไง:** NOMAD ชนะเรื่อง **ปริมาณ** (121 เปเปอร์ รวมสองชุด เทียบ 43 ของเรา) **คำศัพท์มาตรฐาน**
(ชื่อวัสดุ, รหัส ISOS เป็นช่องแยกแทนที่จะฝังในข้อความ) และ **เปิดสาธารณะ**

แยกตามชุดแล้วใช้คนละแบบ
- **manual** — คุณภาพใกล้เคียงงานมือของเรา แต่หยุดที่ปี 2021 และซ้ำกับเราแค่ 1 เปเปอร์
  → ใช้เป็น **ข้อมูลเสริมเพิ่มจำนวนตัวอย่าง** สำหรับคำถามระดับ PCE/องค์ประกอบของเปเปอร์ยุคก่อน 2021 ได้
- **LLM** — ครอบคลุมปีใหม่ซึ่งตรงกับช่วงของเรา แต่สกัดได้ไม่ครบ (คู่ควบคุม 1%, สภาวะทดสอบ 5–13%)
  และข้อ 4.2 พบว่าบันทึกสูตรเคมีผิดอย่างน้อย 2 เปเปอร์ → **ใช้เป็นตัวชี้เป้าว่ามีเปเปอร์อะไรบ้าง
  แล้วกลับไปอ่านต้นฉบับเอง** ไม่ควรใช้ค่าตรง ๆ

ทั้งสองชุดไม่เหมาะกับคำถามหลักของเรา (process → คุณภาพฟิล์ม → ความเสถียร) เพราะช่องที่ต้องใช้
ไม่มีใน schema หรือว่างตามหัวข้อ 5.2–5.5 · และถ้าจะเอามารวม ต้องระวังจุดที่ข้อ 4 และ 5.2 เจอ
(สูตรเคมีผิดบางเปเปอร์, ช่อง PCE end สองความหมาย)

## 6. สรุป (ณ snapshot 12 ต.ค. 2026)

### ข้อ 1 — เปเปอร์ที่ซ้ำ

| | จำนวน |
|---|---|
| NOMAD ทั้งฐาน | ~5.1 หมื่นเครื่อง |
| NOMAD หลังกรองครบ 4 เงื่อนไข | **389 เครื่อง · 121 เปเปอร์** — แยกเป็น manual 206 เครื่อง/47 เปเปอร์ (ปี 2017–2021) · LLM 183 เครื่อง/74 เปเปอร์ (ปี 2021–2026) |
| **ซ้ำกับฐานข้อมูลเรา** | **7 เปเปอร์** (16% ของ 43 เปเปอร์ของเรา) |
| └ แยกตามผู้กรอก | **manual 1 เปเปอร์ · LLM 6 เปเปอร์** — ชุดที่คนกรอกหยุดที่ปี 2021 จึงแทบไม่ทับกับช่วงของเรา |
| มีใน NOMAD แต่หลุดตัวกรอง | 2 เปเปอร์ — LLM บันทึกสูตรไม่มี Cs แต่ทีมเราบันทึกว่ามี → ถ้า NOMAD บันทึกถูกจะซ้ำ **9** |
| เปเปอร์ของเราที่ NOMAD **ไม่มีเลย** | **34 / 43 (79%)** |

หมายเหตุเงื่อนไข 4: ตีความตรงตัว (Efficiency **หรือ** T80 **หรือ** initial+end) ตัดออกแค่ 5 เครื่อง เพราะเกือบทุกเครื่องมี Efficiency
ถ้าต้องการแบบเข้ม (ต้องมีข้อมูลความเสถียร) จะเหลือ 56 เครื่อง · 37 เปเปอร์ — เปลี่ยนได้ที่ตัวแปร `nm["pass4"]` ในข้อ 3

### ข้อ 2 — จุดแข็งของฐานข้อมูลเรา (เรียงตามความสำคัญต่อเป้าหมายโปรเจกต์)

ตัวเทียบหลักคือ **manual** เพราะเป็นงานที่คนอ่านเปเปอร์แล้วกรอกเหมือนกัน ความต่างจึงมาจาก
*สิ่งที่เลือกเก็บ* ไม่ใช่วิธีผลิตข้อมูล · คอลัมน์ LLM แสดงไว้ให้เห็นว่าข้อมูลที่สกัดอัตโนมัติต่างออกไปอย่างไร

| # | จุดแข็ง | ของเรา | NOMAD manual | NOMAD LLM |
|---|---|---|---|---|
| 1 | **ตัวเลขกระบวนการผลิตที่ทำซ้ำในแล็บได้** — อบ NiOx (อุณหภูมิ) · รอบ spin · จังหวะหยด antisolvent | 85% · 89% · 72% | 13% · ไม่มีช่อง · ไม่มีช่อง | 20% · ไม่มีช่อง · ไม่มีช่อง |
| 2 | **ผลวัดฟิล์ม/รอยต่อที่อธิบายกลไก** — trap density · work function/VBM · AFM · มุมสัมผัส | 43% · 46% · 32% · 31% | ไม่มีช่องทั้งหมด | ไม่มีช่องทั้งหมด |
| 3 | **ความเสถียรพร้อมสภาวะทดสอบ** — มีตัวเลข · ระบุความชื้น · ทดสอบหลายสภาวะในเครื่องเดียว | 57% · 39% · 25% | 25% · 2% · ไม่มีช่อง | 13% · 0% · ไม่มีช่อง |
| 4 | **คู่เครื่องควบคุม–เครื่องปรับปรุงในเปเปอร์เดียว** (คิด Δ ได้) | 67% ของเปเปอร์ | 26% ของเปเปอร์ | 1% ของเปเปอร์ |
| 5 | **ข้อมูลผู้ผลิต/ความบริสุทธิ์สารเคมี** | 95% · 68% | 9% · 0% | 0% · 0% |
| 6 | **เปเปอร์ใหม่ที่คนอ่านเอง** — NOMAD ช่วงหลังปี 2021 มาจาก LLM ทั้งหมด | 79% ไม่ซ้ำกับ NOMAD | หยุดที่ปี 2021 | เก็บไม่ครบ/สูตรผิดบางเปเปอร์ (ข้อ 4) |

**ช่องที่ต้องไม่อ้างเกินจริง** — การอบ perovskite ชุด manual ได้ **85–89% ใกล้เคียงของเรา (96%)**
ไม่ใช่จุดแข็งของเรา · และ LLM เก็บความเข้มข้นสารตั้งต้นได้ดีกว่า manual (34% vs 6%)
ตัวเลขรวมที่เคยดูเหมือนเราชนะทุกช่อง เกิดจากการเอาสองชุดมาเฉลี่ยปนกัน

ข้อ 1–4 คือสิ่งที่คำถามหลักของเรา (**process → คุณภาพฟิล์ม → ประสิทธิภาพ/ความเสถียร**) ต้องใช้ และ NOMAD ไม่มีหรือแทบว่าง —
จึงเป็นเหตุผลที่ต้องสร้างฐานข้อมูลเอง ไม่ใช่แค่ดาวน์โหลด NOMAD มาใช้

### ด้านที่ NOMAD ดีกว่า
- **ปริมาณ** ~3 เท่า (121 vs 43 เปเปอร์) · **คำศัพท์มาตรฐาน** (รหัส ISOS เป็นช่องแยก) · **เปิดสาธารณะ** และโตต่อเนื่องด้วย LLM
- **manual** คุณภาพใกล้เคียงงานมือของเรา แต่หยุดที่ปี 2021 และซ้ำกับเราแค่ 1 เปเปอร์ →
  ใช้เสริมจำนวนตัวอย่างสำหรับคำถามระดับ PCE/องค์ประกอบของเปเปอร์ยุคก่อน 2021
- **LLM** ครอบคลุมปีที่ตรงกับเรา แต่สกัดได้ไม่ครบ (คู่ควบคุม 1%) → ใช้เป็นตัวชี้เป้าว่ามีเปเปอร์อะไรบ้าง
  แล้วกลับไปอ่านต้นฉบับเอง ไม่ควรใช้ค่าตรง ๆ
- ถ้ารวมต้องระวัง: สูตรผิดบางเปเปอร์ และช่อง `PCE end` สองความหมาย

### สิ่งที่ฝั่งเราควรแก้ (เจอระหว่างเทียบ)
- **2 เปเปอร์กรอกซ้ำสองชุด** (`solr.202400534`, `acsami.5c16972` — รวม 5 เครื่องเกิน) และชุดของ `solr.202400534` บันทึก Br ไม่ตรงกัน (0.03 vs 0.17)
- **ความหนาฟิล์ม** ยังต่ำ (HTL 39%, perovskite 30%) — จุดอ่อนร่วมของทั้งสองฐาน
- ขนาดเกรนยังปน Top/Bottom (ตามมติ 28 ก.ย. ใช้ได้เฉพาะ Top)
- เปเปอร์ `d3ta05069f` PCE ไม่ตรงกับ NOMAD — ควรเปิดเปเปอร์ยืนยันค่าของเรา